In [1]:
import xarray as xr
import numpy as np
import sys,os
from pathlib import Path
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools.utils import dataarray_healpix_to_equatorial_latlon
from wave_tools import calculate_cross_spectrum, quick_cross_spectrum,remove_annual_cycle
from wave_tools.utils import  get_curve
import intake
import pandas as pd

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
pwd

'/work/mh1498/m301257/code'

In [3]:
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")

In [4]:
ds_cntl = cat.ICON.C5['AMIP_CNTL'].to_dask().sel(time=slice("1980", "1993"))

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/xarray/core/concat.py:546: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


In [5]:
ds_cntl

<xarray.Dataset>
Dimensions:             (cell: 786432, time: 5114, level_full: 26,
                         level_half: 26)
Coordinates:
  * time                (time) datetime64[ns] 1980-01-01 ... 1993-12-31
    healpix             int64 1
  * level_full          (level_full) float64 14.0 21.0 25.0 ... 87.0 89.0 90.0
  * level_half          (level_half) float64 14.0 21.0 25.0 ... 87.0 89.0 90.0
Dimensions without coordinates: cell
Data variables: (12/47)
    cell_elevation      (cell) float64 dask.array<chunksize=(786432,), meta=np.ndarray>
    cell_sea_land_mask  (cell) int32 dask.array<chunksize=(786432,), meta=np.ndarray>
    clivi               (time, cell) float32 dask.array<chunksize=(366, 786432), meta=np.ndarray>
    cllvi               (time, cell) float32 dask.array<chunksize=(366, 786432), meta=np.ndarray>
    hus2m               (time, cell) float32 dask.array<chunksize=(366, 786432), meta=np.ndarray>
    hfls                (time, cell) float32 dask.array<chunksize=(366, 786432), meta=np.ndarray>
    ...                  ...
    ta                  (time, level_full, cell) float32 dask.array<chunksize=(366, 26, 786432), meta=np.ndarray>
    pfull               (time, level_full, cell) float32 dask.array<chunksize=(366, 26, 786432), meta=np.ndarray>
    phalf               (time, level_half, cell) float32 dask.array<chunksize=(366, 26, 786432), meta=np.ndarray>
    zg                  (level_full, cell) float32 dask.array<chunksize=(26, 786432), meta=np.ndarray>
    zghalf              (level_half, cell) float32 dask.array<chunksize=(26, 786432), meta=np.ndarray>
    dzghalf             (level_full, cell) float32 dask.array<chunksize=(26, 786432), meta=np.ndarray>
Attributes:
    history:  Wed May 15 09:39:51 2024: ncatted -O -a ,global,d,, independent...
    NCO:      netCDF Operators version 5.0.6 (Homepage = http://nco.sf.net, C...

In [6]:
# ds_p4k = cat.ICON.C5['AMIP_P4K'].to_dask().sel(time=slice("1980", "1993"))
# ds_4co2 = cat.ICON.C5['AMIP_4CO2'].to_dask().sel(time=slice("1980", "1993"))

In [7]:
#Example mean over january
plvl_list = [5e1, 1e2, 2e2, 2.5e2, 3e2, 3.5e2, 4e2, 5e2, 6e2, 7e2, 8e2, 8.5e2, 9e2, 1e3] # in hPa ... pfull is in Pa
plvl_list



# ds_do = ds_raw.sel(time=slice('1979-01-01', '1979-01-31'))
# ds_plvl = (hacky_plvl_interpolation(ds_do['ta'], ds_do['pfull'] / 100, plvl_list[0])).compute().mean('time')
# for plvl_do in plvl_list[1:]:
#    print(f"# Work on: 1979-01-01 - {plvl_do} hPa", flush=True)
#    # Concat
#    ds_plvl = xr.concat([ds_plvl, (hacky_plvl_interpolation(ds_do['ta'], ds_do['pfull'] / 100, plvl_do)).compute().mean('time')], dim='plev')
#    ds_plvl = ds_plvl.assign_coords(time=pd.Timestamp('1979-01-01')).expand_dims('time')

[50.0,
 100.0,
 200.0,
 250.0,
 300.0,
 350.0,
 400.0,
 500.0,
 600.0,
 700.0,
 800.0,
 850.0,
 900.0,
 1000.0]

In [8]:
def hacky_plvl_interpolation(data_var, data_pfull, plvl_target, unit='hPa', label='pressure level', height_name='level_full'):
    var_height_name_idx = np.where([height_name in dim_name for dim_name in list(data_var.coords)])[0]
    var_height_name = np.array(list(data_var.coords))[var_height_name_idx][0]
    pfull_height_name_idx = np.where([height_name in dim_name for dim_name in list(data_pfull.coords)])[0]
    pfull_height_name = np.array(list(data_pfull.coords))[pfull_height_name_idx][0]
    level_above = (data_pfull > plvl_target).argmax(dim=pfull_height_name).compute() #must be loaded or computed 
    level_below = level_above - 1
    value_above = data_pfull.isel({pfull_height_name: level_above})
    value_below = data_pfull.isel({pfull_height_name: level_below})
    f = (plvl_target - value_below) / (value_above - value_below)
    data_interpolated = (1-f) * data_var.isel({var_height_name: level_below}) + f * data_var.isel({var_height_name: level_above})
    data_interpolated = data_interpolated.drop(pfull_height_name).expand_dims(dim={"plev": [plvl_target,]}, axis=-2)
    data_interpolated['plev'].attrs = {'standard_name': 'plev', 'long_name': label, 'units': unit, 'axis': 'Z'}
    return data_interpolated

## 重要说明：ICON模式层次结构

在ICON模式中：
- **full层 (level_full)**: 温度(ta)、比湿等标量场定义在这些层上
- **half层 (level_half)**: 水平风速(ua, va)定义在这些层上
- **pfull**: full层的气压
- **phalf**: half层的气压

因此，在插值ua和va时，需要：
1. 使用 `phalf` 作为气压数据
2. 设置 `height_name='level_half'` 参数

In [ ]:
# 修正版本：使用level_half作为height_name，因为ua和va在half层上
ds_plvl_cntl = (hacky_plvl_interpolation(ds_cntl['ua'], ds_cntl['phalf'] / 100, plvl_list[0], height_name='level_half')).compute()
for plvl_do in plvl_list[1:]:
   print(f"# Work on: {plvl_do} hPa", flush=True)
   # Concat - 修正变量名从ds_plvl改为ds_plvl_cntl
   ds_plvl_cntl = xr.concat([ds_plvl_cntl, (hacky_plvl_interpolation(ds_cntl['ua'], ds_cntl['phalf'] / 100, plvl_do, height_name='level_half')).compute()], dim='plev')

# 查看结果
print(ds_plvl_cntl)

In [ ]:
# 同样处理va变量
ds_plvl_va_cntl = (hacky_plvl_interpolation(ds_cntl['va'], ds_cntl['phalf'] / 100, plvl_list[0], height_name='level_half')).compute()
for plvl_do in plvl_list[1:]:
   print(f"# Work on va: {plvl_do} hPa", flush=True)
   ds_plvl_va_cntl = xr.concat([ds_plvl_va_cntl, (hacky_plvl_interpolation(ds_cntl['va'], ds_cntl['phalf'] / 100, plvl_do, height_name='level_half')).compute()], dim='plev')

# 查看结果
print(ds_plvl_va_cntl)